Fine-Tuned BERT

In [22]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments
from datasets import Dataset

In [23]:
training_data = pd.read_csv('/sentiment-topic-train.tsv', sep='\t', header=0)
test_data = pd.read_csv('/sentiment-topic-test.tsv', sep='\t')

In [24]:
# Split the training data into training and validation set (same as SVM approach)
train, dev = train_test_split(training_data, test_size=0.1, random_state=0,
                              stratify=training_data[['sentiment']])

# Create mapping for sentiment labels
sentiment_labels = {'negative': 0, 'neutral': 1, 'positive': 2}

# Convert to Hugging Face datasets
train_dataset = Dataset.from_pandas(train)
dev_dataset = Dataset.from_pandas(dev)
test_dataset = Dataset.from_pandas(test_data)

In [25]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# Preprocessing function
def preprocess_function(examples):
    return tokenizer(examples['sentence'], truncation=True, padding='max_length', max_length=128)

# Tokenize and prepare datasets
train_dataset = train_dataset.map(preprocess_function, batched=True)
dev_dataset = dev_dataset.map(preprocess_function, batched=True)
test_dataset = test_dataset.map(preprocess_function, batched=True)

# Set correct format
train_dataset = train_dataset.map(lambda examples: {'labels': sentiment_labels[examples['sentiment']]})
dev_dataset = dev_dataset.map(lambda examples: {'labels': sentiment_labels[examples['sentiment']]})
test_dataset = test_dataset.map(lambda examples: {'labels': sentiment_labels[examples['sentiment']]})

# Set columns format
train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
dev_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])


Map:   0%|          | 0/32200 [00:00<?, ? examples/s]

Map:   0%|          | 0/3578 [00:00<?, ? examples/s]

Map:   0%|          | 0/18 [00:00<?, ? examples/s]

Map:   0%|          | 0/32200 [00:00<?, ? examples/s]

Map:   0%|          | 0/3578 [00:00<?, ? examples/s]

Map:   0%|          | 0/18 [00:00<?, ? examples/s]

In [28]:
# Load the model
model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=3)

# Define metrics function with F1-score for imbalanced classes
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    # Macro F1 treats all classes equally regardless of their frequency
    f1_macro = f1_score(labels, predictions, average='macro')
    # Weighted F1 accounts for class imbalance by weighting by support
    f1_weighted = f1_score(labels, predictions, average='weighted')
    return {
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted
    }

training_args = TrainingArguments(
    output_dir='/content/drive/MyDrive/bert_sentiment_model',
    num_train_epochs=5,              # From a paper
    per_device_train_batch_size=16,
    learning_rate=2e-5,              # From a paper
    evaluation_strategy="epoch",     # Evaluate after each epoch
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",  # Use weighted F1 for imbalanced data
    report_to="none"                 # Disable wandb and all other reporting!
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [29]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    compute_metrics=compute_metrics
)

# Train the model
print("Starting training...")
trainer.train()

Starting training...


Epoch,Training Loss,Validation Loss,F1 Macro,F1 Weighted
1,0.672700,0.663049,0.714467,0.731557
2,0.538300,0.649407,0.730900,0.745575
3,0.391200,0.757479,0.727493,0.745320
4,0.274200,0.909983,0.715638,0.732894
5,0.206200,1.086118,0.716703,0.734683


TrainOutput(global_step=10065, training_loss=0.4342484993894362, metrics={'train_runtime': 3499.6719, 'train_samples_per_second': 46.004, 'train_steps_per_second': 2.876, 'total_flos': 1.0590315063552e+16, 'train_loss': 0.4342484993894362, 'epoch': 5.0})

In [30]:
# Predict on test set
print("Evaluating on test set...")
predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

# Calculate error rate
error_rate = 1 - (preds == labels).mean()
print(f"Error Rate: {error_rate:.4f}")

# Convert numeric predictions back to text labels for report
label_map_reverse = {v: k for k, v in sentiment_labels.items()}
pred_labels = [label_map_reverse[p] for p in preds]
true_labels = [label_map_reverse[l] for l in labels]

# Display classification report
print("\nClassification Report:")
print(classification_report(true_labels, pred_labels))

Evaluating on test set...


Error Rate: 0.1667

Classification Report:
              precision    recall  f1-score   support

    negative       0.71      0.83      0.77         6
     neutral       1.00      0.67      0.80         6
    positive       0.86      1.00      0.92         6

    accuracy                           0.83        18
   macro avg       0.86      0.83      0.83        18
weighted avg       0.86      0.83      0.83        18



In [31]:
# Save the model
print("Saving model...")
trainer.save_model('/content/drive/MyDrive/bert_sentiment_model/final')

Saving model...
